# People Identification in the Wild - Complete Project Notebook

This notebook implements the full project objective end-to-end:

1. Explore and prepare the provided online dataset (including zip handling)
2. Build a face signature gallery from known people
3. Detect faces in a crowd/drone-style image
4. Match each detected face with the query face signature
5. Localize and identify the best match (bounding box + ID)
6. (Optional) Process full videos frame-by-frame

The notebook is intentionally **heavily commented** so each step is easy to understand and explain in your report/presentation.

## 0) Install and import dependencies

We use:
- **DeepFace** for embeddings (face signatures)
- **RetinaFace / OpenCV backends via DeepFace** for face detection
- **OpenCV + Matplotlib** for visualization
- **NumPy/Pandas** for data handling

If you run this in Colab or a fresh environment, uncomment the install lines in the next cell.

In [ ]:
# -----------------------------
# INSTALL (UNCOMMENT IF NEEDED)
# -----------------------------
# !pip install deepface
# !pip install opencv-python matplotlib pandas tqdm

# -----------------------------
# IMPORTS
# -----------------------------
import os
import glob
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from deepface import DeepFace

# Make plots look cleaner
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120

print("Imports loaded successfully.")

## 1) Locate and prepare dataset (including zip exploration)

This cell does three important things:

1. Finds any `.zip` files in the project folder (in case your dataset was added as a zip)
2. Extracts the zip into `open_data_set/` if needed
3. Confirms expected subfolders and prints image counts

This makes the notebook robust even if you share it with someone else who only has the zip file.

In [ ]:
# -----------------------------
# DATASET PATH DISCOVERY + ZIP EXTRACTION
# -----------------------------
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "open_data_set"

# If user dropped zip files in the project root, detect them automatically.
zip_files = sorted(PROJECT_ROOT.glob("*.zip"))

if zip_files:
    print("Found zip file(s):")
    for z in zip_files:
        print(" -", z.name)

    # Extract each zip into DATASET_ROOT. We keep this idempotent: extraction is safe to run again.
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    for z in zip_files:
        print(f"Extracting {z.name} -> {DATASET_ROOT}")
        with zipfile.ZipFile(z, "r") as zip_ref:
            zip_ref.extractall(DATASET_ROOT)
else:
    print("No zip file found in project root. Assuming dataset is already extracted.")

# Expected folders from your dataset package.
expected_folders = [
    "photos_all",        # Crowd/group images
    "photos_all_faces",  # Pre-cropped faces for identities
    "portraits",         # Query-like single-person images
    "trio_cam",
    "trio_gp",
]

print("\nDataset root:", DATASET_ROOT)
for folder in expected_folders:
    folder_path = DATASET_ROOT / folder
    image_files = []
    if folder_path.exists():
        for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
            image_files.extend(folder_path.glob(ext))
    print(f"{folder:16s} -> exists={folder_path.exists()} | images={len(image_files)}")

## 2) Build metadata table from dataset files

We create a table of image paths and inferred identity labels.

- For this dataset, image names are like `a_gp_...jpg`, `k_gp_...jpg`.
- We infer identity from the first token (`a`, `b`, ..., `k`).
- This allows quick filtering for gallery faces and query faces.

In [ ]:
# -----------------------------
# BUILD IMAGE METADATA DATAFRAME
# -----------------------------
def list_images(folder: Path):
    """Return all image files from a folder with common extensions."""
    images = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        images.extend(folder.glob(ext))
    return sorted(images)


def infer_identity_from_filename(path: Path) -> str:
    """
    Infer person identity from filename prefix.
    Example: a_gp_4_ef_06.jpg -> 'a'

    We return uppercase ID labels (A, B, ..., K) to keep display clean.
    """
    token = path.stem.split("_")[0]
    return token.upper()

rows = []

# Gallery candidates: usually already-cropped face images.
gallery_dir = DATASET_ROOT / "photos_all_faces"
for p in list_images(gallery_dir):
    rows.append({
        "path": str(p),
        "split": "gallery",
        "identity": infer_identity_from_filename(p)
    })

# Query candidates: portrait images (single person).
portrait_dir = DATASET_ROOT / "portraits"
for p in list_images(portrait_dir):
    rows.append({
        "path": str(p),
        "split": "query_pool",
        "identity": infer_identity_from_filename(p)
    })

# Scene candidates: crowd/group images where we localize target.
scene_dir = DATASET_ROOT / "photos_all"
for p in list_images(scene_dir):
    rows.append({
        "path": str(p),
        "split": "scene_pool",
        "identity": p.stem.split("_")[0].upper()  # group token, not single identity
    })

meta_df = pd.DataFrame(rows)
print("Total records:", len(meta_df))
print(meta_df["split"].value_counts())

# Show available individual IDs from gallery/query pools.
id_df = meta_df[meta_df["split"].isin(["gallery", "query_pool"])]
print("\nUnique individual IDs:", sorted(id_df["identity"].unique()))
meta_df.head()

## 2.1) Explore dataset composition (important for objective quality)

Before training/matching, always inspect data balance.

This cell shows:
- how many gallery/query images each identity has
- a few random gallery samples

This is useful evidence for your report's data collection section.

In [ ]:
# -----------------------------
# DATASET EXPLORATION (COUNTS + SAMPLE IMAGES)
# -----------------------------
# Count identities in gallery and query pools.
count_table = (
    meta_df[meta_df["split"].isin(["gallery", "query_pool"])]
    .groupby(["split", "identity"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "identity"])
)

display(count_table)

# Plot identity distribution for gallery set.
gallery_counts = count_table[count_table["split"] == "gallery"]
plt.figure(figsize=(8, 4))
plt.bar(gallery_counts["identity"], gallery_counts["count"])
plt.title("Gallery image count per identity")
plt.xlabel("Identity")
plt.ylabel("Number of images")
plt.show()

# Show random gallery examples for quick visual inspection.
sample_gallery = meta_df[meta_df["split"] == "gallery"].sample(8, random_state=42)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (_, row) in zip(axes.flatten(), sample_gallery.iterrows()):
    img = cv2.cvtColor(cv2.imread(row["path"]), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"ID={row['identity']}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3) Choose query image and scene image

- **Query image:** the person we want to find
- **Scene image:** crowd/group image where that person might appear

Tip: choose a query ID that is likely present in the scene token (for example, scene token `AB` likely contains A and B).

In [ ]:
# -----------------------------
# SELECT QUERY + SCENE
# -----------------------------
TARGET_ID = "A"  # Change this to any available ID: A, B, C, ...

# Pick first portrait for TARGET_ID if available, otherwise fallback to gallery image.
query_candidates = meta_df[(meta_df["identity"] == TARGET_ID) & (meta_df["split"] == "query_pool")]
if len(query_candidates) == 0:
    query_candidates = meta_df[(meta_df["identity"] == TARGET_ID) & (meta_df["split"] == "gallery")]

if len(query_candidates) == 0:
    raise ValueError(f"No query image found for TARGET_ID={TARGET_ID}")

query_path = query_candidates.iloc[0]["path"]

# Choose a scene image that likely contains target ID (based on group token in filename prefix).
scene_candidates = meta_df[(meta_df["split"] == "scene_pool") & (meta_df["identity"].str.contains(TARGET_ID, na=False))]
if len(scene_candidates) == 0:
    scene_candidates = meta_df[meta_df["split"] == "scene_pool"]

scene_path = scene_candidates.iloc[0]["path"]

print("Selected TARGET_ID:", TARGET_ID)
print("Query image:", query_path)
print("Scene image:", scene_path)

# Visual sanity check.
query_img = cv2.cvtColor(cv2.imread(query_path), cv2.COLOR_BGR2RGB)
scene_img = cv2.cvtColor(cv2.imread(scene_path), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(query_img)
axes[0].set_title(f"Query (ID={TARGET_ID})")
axes[0].axis("off")

axes[1].imshow(scene_img)
axes[1].set_title("Scene / Crowd image")
axes[1].axis("off")
plt.show()

## 4) Build gallery signatures (embeddings)

This corresponds to objective steps:
- Crop/use face images
- Extract signatures `s1, s2, ..., sn`

Here we use `photos_all_faces` as face crops, so we can run embedding extraction with `detector_backend='skip'` (faster and cleaner).

In [ ]:
# -----------------------------
# SIGNATURE EXTRACTION HELPERS
# -----------------------------
MODEL_NAME = "ArcFace"          # Strong default for face recognition
DISTANCE_METRIC = "cosine"      # We'll compute cosine distance manually


def get_embedding(image_path: str, model_name: str = MODEL_NAME, detector_backend: str = "skip"):
    """
    Return one embedding vector for a single image.

    Why detector_backend='skip'?
    - For pre-cropped face images (gallery/query), detection is unnecessary and can fail less often.
    """
    try:
        reps = DeepFace.represent(
            img_path=image_path,
            model_name=model_name,
            detector_backend=detector_backend,
            enforce_detection=False
        )
        if len(reps) == 0:
            return None
        # DeepFace returns a list of dicts; each dict has an 'embedding' vector.
        emb = np.array(reps[0]["embedding"], dtype=np.float32)
        return emb
    except Exception as e:
        print(f"Embedding failed for {image_path}: {e}")
        return None


def cosine_distance(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> float:
    """Compute cosine distance = 1 - cosine_similarity."""
    a_norm = np.linalg.norm(a) + eps
    b_norm = np.linalg.norm(b) + eps
    return float(1.0 - np.dot(a, b) / (a_norm * b_norm))


# Build gallery signatures (we use a subset per identity for speed in classroom runs).
gallery_df = meta_df[meta_df["split"] == "gallery"].copy()

# Optional downsample: keep up to N images per identity for faster runs.
MAX_PER_ID = 30
gallery_df = gallery_df.groupby("identity", group_keys=False).head(MAX_PER_ID).reset_index(drop=True)

embeddings = []
for _, row in tqdm(gallery_df.iterrows(), total=len(gallery_df), desc="Building gallery embeddings"):
    emb = get_embedding(row["path"], detector_backend="skip")
    embeddings.append(emb)

gallery_df["embedding"] = embeddings
gallery_df = gallery_df[gallery_df["embedding"].notnull()].reset_index(drop=True)

print("Gallery size after embedding extraction:", len(gallery_df))
gallery_df.head()

## 5) Extract query signature `qs`

This is objective step 4: create the query embedding (`qs`) from the query face image.

In [ ]:
# -----------------------------
# QUERY SIGNATURE
# -----------------------------
query_embedding = get_embedding(query_path, detector_backend="skip")

if query_embedding is None:
    raise RuntimeError("Could not extract query embedding. Try another query image.")

print("Query embedding length:", len(query_embedding))

## 6) Detect all faces in the scene image and crop them

This directly matches objective steps 1 and 2:

1. Detect all faces in input scene image (`n` faces)
2. Crop all detected faces (`n` face crops)

We use DeepFace face extraction with RetinaFace backend for robust detection.

In [ ]:
# -----------------------------
# FACE DETECTION + CROP IN SCENE IMAGE
# -----------------------------
scene_bgr = cv2.imread(scene_path)
if scene_bgr is None:
    raise FileNotFoundError(f"Could not read scene image: {scene_path}")

scene_h, scene_w = scene_bgr.shape[:2]


def looks_like_full_frame_fallback(area: dict, img_w: int, img_h: int) -> bool:
    """
    DeepFace with enforce_detection=False may return the whole image as one "face"
    when no actual face is detected. We explicitly reject that case.
    """
    x, y, w, h = int(area["x"]), int(area["y"]), int(area["w"]), int(area["h"])
    covers_most_width = w >= int(0.9 * img_w)
    covers_most_height = h >= int(0.9 * img_h)
    near_top_left = (x <= int(0.05 * img_w)) and (y <= int(0.05 * img_h))
    return covers_most_width and covers_most_height and near_top_left


def detect_scene_faces_robust(base_bgr: np.ndarray):
    """
    Two-pass detection strategy for distant/small faces:
    1) detect on original image
    2) if empty, detect on 2x upscaled image

    We use enforce_detection=True so no fallback "full-image face" is produced.
    """

    def run_detection(img_input, scale_factor: float):
        try:
            raw_faces = DeepFace.extract_faces(
                img_path=img_input,
                detector_backend="retinaface",
                enforce_detection=True,
                align=True,
            )
        except Exception:
            return []

        cleaned = []
        for item in raw_faces:
            area = item["facial_area"]

            # Convert coordinates back to original scene if detection ran on upscaled image.
            x = int(area["x"] / scale_factor)
            y = int(area["y"] / scale_factor)
            w = int(area["w"] / scale_factor)
            h = int(area["h"] / scale_factor)

            # Clamp box coordinates to valid image boundaries.
            x = max(0, min(x, scene_w - 1))
            y = max(0, min(y, scene_h - 1))
            w = max(1, min(w, scene_w - x))
            h = max(1, min(h, scene_h - y))

            area_fixed = {"x": x, "y": y, "w": w, "h": h}

            # Reject impossible/fallback boxes.
            if looks_like_full_frame_fallback(area_fixed, scene_w, scene_h):
                continue

            # Reject very tiny noise boxes but keep distant faces.
            if w < 12 or h < 12:
                continue

            # Crop directly from original image so downstream embedding uses full-quality crop.
            crop_bgr = scene_bgr[y : y + h, x : x + w]
            if crop_bgr.size == 0:
                continue
            crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

            cleaned.append(
                {
                    "face": crop_rgb.astype(np.float32) / 255.0,
                    "facial_area": area_fixed,
                    "confidence": item.get("confidence", None),
                }
            )

        return cleaned

    # Pass 1: original resolution
    faces = run_detection(base_bgr, scale_factor=1.0)

    # Pass 2: if no detections, try 2x upscaling (helps tiny/far faces)
    if len(faces) == 0:
        upscaled = cv2.resize(base_bgr, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
        faces = run_detection(upscaled, scale_factor=2.0)

    return faces


scene_faces = detect_scene_faces_robust(scene_bgr)
print("Detected faces in scene after filtering:", len(scene_faces))

if len(scene_faces) == 0:
    raise RuntimeError(
        "No reliable faces detected in this scene. Try a different scene image, "
        "or increase resolution before detection."
    )

# Quick visualization of detected face crops
num_show = min(8, len(scene_faces))
fig, axes = plt.subplots(1, num_show, figsize=(3 * num_show, 3))
if num_show == 1:
    axes = [axes]

for i in range(num_show):
    crop_rgb = (scene_faces[i]["face"] * 255).astype(np.uint8)
    axes[i].imshow(crop_rgb)
    axes[i].set_title(f"Face #{i}")
    axes[i].axis("off")

if num_show > 0:
    plt.tight_layout()
    plt.show()

## 6.1) Improved detector for far/small faces (multi-backend + tiles)

If far faces are missed, run this cell before matching.

This version:
- Tries multiple detector backends (`retinaface`, `mtcnn`, `opencv`)
- Runs at multiple scales
- Runs on overlapping tiles to recover tiny faces
- Merges duplicates via NMS

It replaces `scene_faces` with better detections for the next matching step.

In [ ]:
# -----------------------------
# IMPROVED FAR-FACE DETECTOR (OVERRIDES scene_faces)
# -----------------------------
DETECTOR_BACKENDS = ["retinaface", "mtcnn", "opencv"]
UPSCALE_FACTORS = [1.0, 1.5, 2.0]


def _iou_xywh(a, b):
    ax1, ay1, aw, ah = a
    bx1, by1, bw, bh = b
    ax2, ay2 = ax1 + aw, ay1 + ah
    bx2, by2 = bx1 + bw, by1 + bh
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    union = (aw * ah) + (bw * bh) - inter + 1e-8
    return inter / union


def _nms_xywh(dets, iou_thresh=0.35):
    if not dets:
        return []
    dets = sorted(dets, key=lambda d: d["score"], reverse=True)
    keep = []
    while dets:
        best = dets.pop(0)
        keep.append(best)
        filtered = []
        for d in dets:
            iou = _iou_xywh((best["x"], best["y"], best["w"], best["h"]), (d["x"], d["y"], d["w"], d["h"]))
            if iou < iou_thresh:
                filtered.append(d)
        dets = filtered
    return keep


def _tiles_for_image(img_w, img_h, tile_ratio=0.5, overlap=0.35):
    tile_w = int(img_w * tile_ratio)
    tile_h = int(img_h * tile_ratio)
    step_x = max(1, int(tile_w * (1 - overlap)))
    step_y = max(1, int(tile_h * (1 - overlap)))

    tiles = []
    y = 0
    while y < img_h:
        x = 0
        while x < img_w:
            x2 = min(img_w, x + tile_w)
            y2 = min(img_h, y + tile_h)
            x1 = max(0, x2 - tile_w)
            y1 = max(0, y2 - tile_h)
            tiles.append((x1, y1, x2, y2))
            if x2 == img_w:
                break
            x += step_x
        if y2 == img_h:
            break
        y += step_y
    return tiles


def detect_scene_faces_robust(base_bgr):
    h0, w0 = base_bgr.shape[:2]
    all_dets = []
    backend_hits = {b: 0 for b in DETECTOR_BACKENDS}

    for backend in DETECTOR_BACKENDS:
        for scale in UPSCALE_FACTORS:
            if scale == 1.0:
                scaled = base_bgr
            else:
                scaled = cv2.resize(base_bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

            hs, ws = scaled.shape[:2]
            regions = [(0, 0, ws, hs)]
            regions.extend(_tiles_for_image(ws, hs, tile_ratio=0.5, overlap=0.35))

            for x1, y1, x2, y2 in regions:
                patch = scaled[y1:y2, x1:x2]
                if patch.size == 0:
                    continue
                try:
                    raw_faces = DeepFace.extract_faces(
                        img_path=patch,
                        detector_backend=backend,
                        enforce_detection=True,
                        align=True,
                    )
                except Exception:
                    continue

                for item in raw_faces:
                    area = item["facial_area"]
                    sx = x1 + int(area["x"])
                    sy = y1 + int(area["y"])
                    sw = int(area["w"])
                    sh = int(area["h"])

                    ox = int(sx / scale)
                    oy = int(sy / scale)
                    ow = int(sw / scale)
                    oh = int(sh / scale)

                    ox = max(0, min(ox, w0 - 1))
                    oy = max(0, min(oy, h0 - 1))
                    ow = max(1, min(ow, w0 - ox))
                    oh = max(1, min(oh, h0 - oy))

                    if ow < 10 or oh < 10:
                        continue

                    aspect = ow / (oh + 1e-8)
                    if aspect < 0.45 or aspect > 1.8:
                        continue

                    conf = item.get("confidence", 0.5)
                    conf = 0.5 if conf is None else float(conf)

                    all_dets.append({
                        "x": ox, "y": oy, "w": ow, "h": oh,
                        "score": conf,
                        "backend": backend,
                    })
                    backend_hits[backend] += 1

    merged = _nms_xywh(all_dets, iou_thresh=0.35)

    faces = []
    for d in merged:
        x, y, w, h = d["x"], d["y"], d["w"], d["h"]
        crop_bgr = base_bgr[y:y+h, x:x+w]
        if crop_bgr.size == 0:
            continue
        crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
        faces.append({
            "face": crop_rgb.astype(np.float32) / 255.0,
            "facial_area": {"x": x, "y": y, "w": w, "h": h},
            "confidence": d["score"],
            "backend": d["backend"],
        })

    print("Raw detections by backend:", backend_hits)
    print("Raw detections total:", len(all_dets))
    print("Detections after NMS:", len(faces))
    return faces


scene_faces = detect_scene_faces_robust(scene_bgr)
print("Improved scene_faces count:", len(scene_faces))

if len(scene_faces) == 0:
    raise RuntimeError("No reliable faces detected. Try a different scene frame or a closer shot.")

num_show = min(8, len(scene_faces))
fig, axes = plt.subplots(1, num_show, figsize=(3 * num_show, 3))
if num_show == 1:
    axes = [axes]

for i in range(num_show):
    crop_rgb = (scene_faces[i]["face"] * 255).astype(np.uint8)
    axes[i].imshow(crop_rgb)
    axes[i].set_title(f"Face #{i} ({scene_faces[i].get('backend','n/a')})")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## 6.2) Explicit detector debug view (each backend x each scale)

This diagnostic section shows exactly what each detector backend returns at each scale:

- Annotated scene image (bounding boxes)
- Cropped face thumbnails

Use this to compare detector behavior and verify whether a backend/scale is genuinely finding faces or not.

In [ ]:
# -----------------------------
# EXPLICIT PER-BACKEND / PER-SCALE VISUALIZATION
# -----------------------------
# NOTE:
# - This is a diagnostic cell for interpretability.
# - It intentionally runs FULL-IMAGE detection only (no tiling), so you can see raw
#   backend/scale behavior directly.
# - The robust pipeline in 6.1 remains the main detector for matching.

MAX_CROPS_TO_SHOW = 8


def detect_full_image_for_combo(base_bgr, backend: str, scale: float):
    """Run one detector backend at one scale on the full image only."""
    h0, w0 = base_bgr.shape[:2]

    if scale == 1.0:
        scaled = base_bgr
    else:
        scaled = cv2.resize(base_bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

    try:
        raw_faces = DeepFace.extract_faces(
            img_path=scaled,
            detector_backend=backend,
            enforce_detection=True,
            align=True,
        )
    except Exception:
        return []

    detections = []
    for item in raw_faces:
        area = item["facial_area"]

        # Map coordinates from scaled image back to original scene coordinates.
        x = int(area["x"] / scale)
        y = int(area["y"] / scale)
        w = int(area["w"] / scale)
        h = int(area["h"] / scale)

        # Clamp to bounds.
        x = max(0, min(x, w0 - 1))
        y = max(0, min(y, h0 - 1))
        w = max(1, min(w, w0 - x))
        h = max(1, min(h, h0 - y))

        # Basic noise filtering (keep fairly permissive for inspection).
        if w < 10 or h < 10:
            continue

        conf = item.get("confidence", None)
        conf = -1.0 if conf is None else float(conf)

        detections.append({"x": x, "y": y, "w": w, "h": h, "conf": conf})

    return detections


# Build a table of raw detections for each backend x scale combination.
combo_rows = []
combo_outputs = []

for backend in DETECTOR_BACKENDS:
    for scale in UPSCALE_FACTORS:
        dets = detect_full_image_for_combo(scene_bgr, backend=backend, scale=scale)
        combo_rows.append({
            "backend": backend,
            "scale": scale,
            "num_detections": len(dets),
        })
        combo_outputs.append((backend, scale, dets))

combo_df = pd.DataFrame(combo_rows)
display(combo_df)

# Visualize each combination separately for clarity.
scene_rgb = cv2.cvtColor(scene_bgr, cv2.COLOR_BGR2RGB)

for backend, scale, dets in combo_outputs:
    vis = scene_rgb.copy()

    for d in dets:
        x, y, w, h = d["x"], d["y"], d["w"], d["h"]
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)

    plt.figure(figsize=(10, 5))
    plt.imshow(vis)
    plt.title(f"Backend={backend} | Scale={scale} | Detections={len(dets)}")
    plt.axis("off")
    plt.show()

    # Show cropped faces for this combo.
    if len(dets) == 0:
        print(f"No detections for backend={backend}, scale={scale}\n")
        continue

    show_n = min(MAX_CROPS_TO_SHOW, len(dets))
    fig, axes = plt.subplots(1, show_n, figsize=(3 * show_n, 3))
    if show_n == 1:
        axes = [axes]

    for i in range(show_n):
        d = dets[i]
        x, y, w, h = d["x"], d["y"], d["w"], d["h"]
        crop = scene_rgb[y:y+h, x:x+w]
        axes[i].imshow(crop)
        axes[i].set_title(f"#{i} conf={d['conf']:.2f}")
        axes[i].axis("off")

    plt.suptitle(f"Cropped faces | Backend={backend}, Scale={scale}", y=1.02)
    plt.tight_layout()
    plt.show()

## 6.3) Benchmark additional detector backends and recognition models

Yes — based on the repos you shared, we can test more than one model/backend.

This benchmark compares:
- **Detector backends** (how many faces they detect + runtime)
- **Recognition models** (best query distance + runtime on detected faces)

This gives you evidence for model selection in your report.

In [ ]:
# -----------------------------
# BACKEND + MODEL BENCHMARK
# -----------------------------
import time

# Recognition models supported by DeepFace and relevant to your shared repos:
# - ArcFace / Buffalo_L (InsightFace family)
# - GhostFaceNet (GhostFaceNets repo)
# - SFace / Facenet512 / Dlib as additional baselines
MODEL_CANDIDATES = [
    "ArcFace",
    "Buffalo_L",
    "GhostFaceNet",
    "SFace",
    "Facenet512",
    "Dlib",
]

# Detector backends to compare. Some may fail depending on local dependencies.
DETECTOR_BACKEND_CANDIDATES = [
    "retinaface",
    "mtcnn",
    "opencv",
    "mediapipe",
    "ssd",
    "dlib",
]

# Scales used for detector benchmarking (full-image mode for fair quick comparison).
BENCH_SCALES = [1.0, 1.5, 2.0]


def _safe_query_embedding(model_name: str):
    """Get query embedding for one recognition model, safely."""
    try:
        t0 = time.time()
        rep = DeepFace.represent(
            img_path=query_path,
            model_name=model_name,
            detector_backend="skip",
            enforce_detection=False,
        )
        if len(rep) == 0:
            return None, None, "empty_embedding"
        emb = np.array(rep[0]["embedding"], dtype=np.float32)
        return emb, time.time() - t0, None
    except Exception as e:
        return None, None, str(e)


# Fallback detector helper if 6.2 cell was not run.
if "detect_full_image_for_combo" not in globals():
    def detect_full_image_for_combo(base_bgr, backend: str, scale: float):
        h0, w0 = base_bgr.shape[:2]
        if scale == 1.0:
            scaled = base_bgr
        else:
            scaled = cv2.resize(base_bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

        try:
            raw_faces = DeepFace.extract_faces(
                img_path=scaled,
                detector_backend=backend,
                enforce_detection=True,
                align=True,
            )
        except Exception:
            return []

        out = []
        for item in raw_faces:
            area = item["facial_area"]
            x = int(area["x"] / scale)
            y = int(area["y"] / scale)
            w = int(area["w"] / scale)
            h = int(area["h"] / scale)
            x = max(0, min(x, w0 - 1))
            y = max(0, min(y, h0 - 1))
            w = max(1, min(w, w0 - x))
            h = max(1, min(h, h0 - y))
            if w < 10 or h < 10:
                continue
            out.append({"x": x, "y": y, "w": w, "h": h})
        return out


# -----------------------------
# 1) DETECTOR BACKEND BENCHMARK
# -----------------------------
detector_rows = []
backend_face_boxes = {}

for backend in DETECTOR_BACKEND_CANDIDATES:
    t0 = time.time()
    best_boxes = []
    error_msg = None

    try:
        # For each backend, take the scale that gives the most detections.
        for sc in BENCH_SCALES:
            boxes = detect_full_image_for_combo(scene_bgr, backend=backend, scale=sc)
            if len(boxes) > len(best_boxes):
                best_boxes = boxes
    except Exception as e:
        error_msg = str(e)

    elapsed = time.time() - t0
    backend_face_boxes[backend] = best_boxes

    detector_rows.append(
        {
            "backend": backend,
            "num_faces_best_scale": len(best_boxes),
            "runtime_sec": round(elapsed, 3),
            "error": error_msg,
        }
    )

detector_bench_df = pd.DataFrame(detector_rows).sort_values(
    ["num_faces_best_scale", "runtime_sec"], ascending=[False, True]
)
print("Detector backend benchmark:")
display(detector_bench_df)

# Pick the backend that found most faces.
best_backend_row = detector_bench_df.iloc[0]
BEST_BENCH_BACKEND = best_backend_row["backend"]
bench_boxes = backend_face_boxes.get(BEST_BENCH_BACKEND, [])
print(f"Using backend '{BEST_BENCH_BACKEND}' for model benchmark with {len(bench_boxes)} face crop(s).")

if len(bench_boxes) == 0:
    raise RuntimeError("No detector backend found faces for this scene; try a different scene image.")


# -----------------------------
# 2) RECOGNITION MODEL BENCHMARK
# -----------------------------
model_rows = []

# Pre-crop scene faces once from the chosen backend.
scene_crops_rgb = []
for b in bench_boxes:
    x, y, w, h = b["x"], b["y"], b["w"], b["h"]
    crop_bgr = scene_bgr[y:y+h, x:x+w]
    if crop_bgr.size == 0:
        continue
    scene_crops_rgb.append(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))

for model_name in MODEL_CANDIDATES:
    q_emb, q_time, q_err = _safe_query_embedding(model_name)
    if q_emb is None:
        model_rows.append(
            {
                "model": model_name,
                "best_query_distance": np.nan,
                "num_scene_faces_used": 0,
                "query_embed_sec": None,
                "scene_embed_total_sec": None,
                "error": q_err,
            }
        )
        continue

    best_dist = 999.0
    used = 0
    t_scene0 = time.time()

    for crop_rgb in scene_crops_rgb:
        try:
            rep = DeepFace.represent(
                img_path=crop_rgb,
                model_name=model_name,
                detector_backend="skip",
                enforce_detection=False,
            )
            if len(rep) == 0:
                continue
            emb = np.array(rep[0]["embedding"], dtype=np.float32)
            dist = cosine_distance(q_emb, emb)
            best_dist = min(best_dist, dist)
            used += 1
        except Exception:
            continue

    t_scene = time.time() - t_scene0

    model_rows.append(
        {
            "model": model_name,
            "best_query_distance": (best_dist if used > 0 else np.nan),
            "num_scene_faces_used": used,
            "query_embed_sec": round(q_time, 3) if q_time is not None else None,
            "scene_embed_total_sec": round(t_scene, 3),
            "error": None if used > 0 else "no_valid_scene_embeddings",
        }
    )

model_bench_df = pd.DataFrame(model_rows).sort_values("best_query_distance", na_position="last")
print("Recognition model benchmark (lower best_query_distance is better):")
display(model_bench_df)

# Optional quick recommendation based on this single-scene benchmark.
valid = model_bench_df[model_bench_df["best_query_distance"].notnull()]
if len(valid) > 0:
    best_model = valid.iloc[0]["model"]
    print(f"Suggested model for this scene: {best_model}")

## 6.4) Fixed test setup: OpenCV detector at scale 2.0

This section benchmarks recognition models using your chosen fixed detector configuration:
- backend = `opencv`
- scale = `2.0`

Use this when you want a fair same-detector comparison across recognition models.

In [ ]:
# -----------------------------
# FIXED DETECTOR (opencv, scale=2.0) + RECOGNITION MODEL TEST
# -----------------------------
import time

FIXED_DETECTOR_BACKEND = "opencv"
FIXED_DETECTOR_SCALE = 2.0

MODEL_CANDIDATES = [
    "ArcFace",
    "Buffalo_L",
    "GhostFaceNet",
    "SFace",
    "Facenet512",
    "Dlib",
]


def _safe_query_embedding(model_name: str):
    try:
        t0 = time.time()
        rep = DeepFace.represent(
            img_path=query_path,
            model_name=model_name,
            detector_backend="skip",
            enforce_detection=False,
        )
        if len(rep) == 0:
            return None, None, "empty_embedding"
        emb = np.array(rep[0]["embedding"], dtype=np.float32)
        return emb, time.time() - t0, None
    except Exception as e:
        return None, None, str(e)


# Use helper from 6.2 if available, otherwise define a local fallback.
if "detect_full_image_for_combo" in globals():
    fixed_boxes = detect_full_image_for_combo(
        scene_bgr,
        backend=FIXED_DETECTOR_BACKEND,
        scale=FIXED_DETECTOR_SCALE,
    )
else:
    h0, w0 = scene_bgr.shape[:2]
    scaled = cv2.resize(scene_bgr, None, fx=FIXED_DETECTOR_SCALE, fy=FIXED_DETECTOR_SCALE, interpolation=cv2.INTER_CUBIC)
    try:
        raw_faces = DeepFace.extract_faces(
            img_path=scaled,
            detector_backend=FIXED_DETECTOR_BACKEND,
            enforce_detection=True,
            align=True,
        )
    except Exception:
        raw_faces = []

    fixed_boxes = []
    for item in raw_faces:
        area = item["facial_area"]
        x = int(area["x"] / FIXED_DETECTOR_SCALE)
        y = int(area["y"] / FIXED_DETECTOR_SCALE)
        w = int(area["w"] / FIXED_DETECTOR_SCALE)
        h = int(area["h"] / FIXED_DETECTOR_SCALE)
        x = max(0, min(x, w0 - 1))
        y = max(0, min(y, h0 - 1))
        w = max(1, min(w, w0 - x))
        h = max(1, min(h, h0 - y))
        if w >= 10 and h >= 10:
            fixed_boxes.append({"x": x, "y": y, "w": w, "h": h})

print(f"Fixed detector setup -> backend={FIXED_DETECTOR_BACKEND}, scale={FIXED_DETECTOR_SCALE}")
print(f"Detected faces: {len(fixed_boxes)}")

if len(fixed_boxes) == 0:
    raise RuntimeError("OpenCV at scale 2.0 found 0 faces on this scene. Try another scene image.")

# Show fixed detector boxes.
vis = cv2.cvtColor(scene_bgr.copy(), cv2.COLOR_BGR2RGB)
for b in fixed_boxes:
    x, y, w, h = b["x"], b["y"], b["w"], b["h"]
    cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)

plt.figure(figsize=(10, 5))
plt.imshow(vis)
plt.title(f"Fixed detector boxes | {FIXED_DETECTOR_BACKEND} @ scale {FIXED_DETECTOR_SCALE}")
plt.axis("off")
plt.show()

# Crop scene faces once from fixed detections.
scene_crops_rgb = []
for b in fixed_boxes:
    x, y, w, h = b["x"], b["y"], b["w"], b["h"]
    crop_bgr = scene_bgr[y:y+h, x:x+w]
    if crop_bgr.size == 0:
        continue
    scene_crops_rgb.append(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))

# Benchmark recognition models under fixed detector setup.
rows = []
for model_name in MODEL_CANDIDATES:
    q_emb, q_time, q_err = _safe_query_embedding(model_name)
    if q_emb is None:
        rows.append({
            "model": model_name,
            "best_query_distance": np.nan,
            "num_scene_faces_used": 0,
            "query_embed_sec": None,
            "scene_embed_total_sec": None,
            "detector_backend": FIXED_DETECTOR_BACKEND,
            "detector_scale": FIXED_DETECTOR_SCALE,
            "error": q_err,
        })
        continue

    best_dist = 999.0
    used = 0
    t0 = time.time()

    for crop_rgb in scene_crops_rgb:
        try:
            rep = DeepFace.represent(
                img_path=crop_rgb,
                model_name=model_name,
                detector_backend="skip",
                enforce_detection=False,
            )
            if len(rep) == 0:
                continue
            emb = np.array(rep[0]["embedding"], dtype=np.float32)
            dist = cosine_distance(q_emb, emb)
            best_dist = min(best_dist, dist)
            used += 1
        except Exception:
            continue

    rows.append({
        "model": model_name,
        "best_query_distance": best_dist if used > 0 else np.nan,
        "num_scene_faces_used": used,
        "query_embed_sec": round(q_time, 3) if q_time is not None else None,
        "scene_embed_total_sec": round(time.time() - t0, 3),
        "detector_backend": FIXED_DETECTOR_BACKEND,
        "detector_scale": FIXED_DETECTOR_SCALE,
        "error": None if used > 0 else "no_valid_scene_embeddings",
    })

fixed_model_bench_df = pd.DataFrame(rows).sort_values("best_query_distance", na_position="last")
print("Recognition model comparison with fixed detector setup:")
display(fixed_model_bench_df)

valid = fixed_model_bench_df[fixed_model_bench_df["best_query_distance"].notnull()]
if len(valid) > 0:
    print(f"Best model in this fixed test: {valid.iloc[0]['model']}")

## 6.5) Show detected face images and recognized IDs

This section explicitly demonstrates how ID recognition works:

1. For each detected face crop in the scene, extract an embedding
2. Compare it to gallery embeddings (same recognition model)
3. Assign the nearest gallery identity as the predicted ID
4. Visualize:
   - Annotated scene image with all predicted IDs
   - Face crops with predicted ID and nearest distance

In [ ]:
# -----------------------------
# VISUALIZE DETECTED FACES + PREDICTED IDs
# -----------------------------
# Pick recognition model for this visualization.
# If fixed benchmark ran, use its best model; otherwise fallback to ArcFace.
if "fixed_model_bench_df" in globals() and len(fixed_model_bench_df) > 0:
    valid_rows = fixed_model_bench_df[fixed_model_bench_df["best_query_distance"].notnull()]
    RECOG_MODEL_FOR_VIS = valid_rows.iloc[0]["model"] if len(valid_rows) > 0 else "ArcFace"
else:
    RECOG_MODEL_FOR_VIS = "ArcFace"

print(f"Using recognition model for ID visualization: {RECOG_MODEL_FOR_VIS}")

# Choose boxes source:
# - Prefer fixed OpenCV@2.0 boxes if available
# - Else fallback to scene_faces from robust detector
if "fixed_boxes" in globals() and len(fixed_boxes) > 0:
    vis_boxes = [{"x": b["x"], "y": b["y"], "w": b["w"], "h": b["h"]} for b in fixed_boxes]
elif "scene_faces" in globals() and len(scene_faces) > 0:
    vis_boxes = [
        {
            "x": int(f["facial_area"]["x"]),
            "y": int(f["facial_area"]["y"]),
            "w": int(f["facial_area"]["w"]),
            "h": int(f["facial_area"]["h"]),
        }
        for f in scene_faces
    ]
else:
    raise RuntimeError("No detections available. Run section 6/6.1/6.4 first.")

# Build gallery embeddings for the SAME recognition model.
# We cache by model to avoid recomputing if cell is rerun.
if "gallery_embeddings_by_model" not in globals():
    gallery_embeddings_by_model = {}

if RECOG_MODEL_FOR_VIS not in gallery_embeddings_by_model:
    print("Building gallery embeddings for model:", RECOG_MODEL_FOR_VIS)
    g_rows = []
    for _, row in tqdm(gallery_df.iterrows(), total=len(gallery_df), desc="Gallery embeddings (ID vis)"):
        try:
            rep = DeepFace.represent(
                img_path=row["path"],
                model_name=RECOG_MODEL_FOR_VIS,
                detector_backend="skip",
                enforce_detection=False,
            )
            if len(rep) == 0:
                continue
            emb = np.array(rep[0]["embedding"], dtype=np.float32)
            g_rows.append({"identity": row["identity"], "embedding": emb})
        except Exception:
            continue

    if len(g_rows) == 0:
        raise RuntimeError("Failed to build gallery embeddings for visualization model.")

    gallery_embeddings_by_model[RECOG_MODEL_FOR_VIS] = pd.DataFrame(g_rows)

gallery_vis_df = gallery_embeddings_by_model[RECOG_MODEL_FOR_VIS]

# Predict ID for each detected box.
id_results = []
for i, b in enumerate(vis_boxes):
    x, y, w, h = b["x"], b["y"], b["w"], b["h"]
    crop_bgr = scene_bgr[y:y+h, x:x+w]
    if crop_bgr.size == 0:
        continue

    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

    try:
        rep = DeepFace.represent(
            img_path=crop_rgb,
            model_name=RECOG_MODEL_FOR_VIS,
            detector_backend="skip",
            enforce_detection=False,
        )
        if len(rep) == 0:
            continue
        emb = np.array(rep[0]["embedding"], dtype=np.float32)
    except Exception:
        continue

    # Nearest-gallery identity.
    dists = gallery_vis_df["embedding"].apply(lambda e: cosine_distance(emb, e))
    best_idx = int(dists.idxmin())
    pred_id = gallery_vis_df.loc[best_idx, "identity"]
    best_dist = float(dists.loc[best_idx])

    id_results.append(
        {
            "det_idx": i,
            "predicted_id": pred_id,
            "nearest_distance": best_dist,
            "x": x,
            "y": y,
            "w": w,
            "h": h,
            "crop_rgb": crop_rgb,
        }
    )

id_results_df = pd.DataFrame([{k: v for k, v in r.items() if k != "crop_rgb"} for r in id_results])
print("Detected faces with predicted IDs:")
display(id_results_df.sort_values("nearest_distance"))

if len(id_results) == 0:
    raise RuntimeError("No valid ID predictions produced.")

# 1) Annotated full scene with all predicted IDs.
scene_annot = scene_bgr.copy()
for r in id_results:
    x, y, w, h = r["x"], r["y"], r["w"], r["h"]
    label = f"ID={r['predicted_id']} d={r['nearest_distance']:.3f}"
    cv2.rectangle(scene_annot, (x, y), (x + w, y + h), (0, 255, 0), 2)
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
    cv2.rectangle(scene_annot, (x, max(0, y - 22)), (x + tw + 6, y), (0, 255, 0), -1)
    cv2.putText(scene_annot, label, (x + 3, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(scene_annot, cv2.COLOR_BGR2RGB))
plt.title(f"All detected faces with predicted IDs ({RECOG_MODEL_FOR_VIS})")
plt.axis("off")
plt.show()

# Save annotated all-ID image.
all_ids_output_path = str(PROJECT_ROOT / "output_all_detected_faces_with_ids.jpg")
cv2.imwrite(all_ids_output_path, scene_annot)
print("Saved annotated all-ID output:", all_ids_output_path)

# 2) Show each face crop and predicted ID explicitly.
max_show = min(12, len(id_results))
fig, axes = plt.subplots(1, max_show, figsize=(3 * max_show, 3))
if max_show == 1:
    axes = [axes]

id_results_sorted = sorted(id_results, key=lambda r: r["nearest_distance"])
for ax, r in zip(axes, id_results_sorted[:max_show]):
    ax.imshow(r["crop_rgb"])
    ax.set_title(f"Det#{r['det_idx']}\nID={r['predicted_id']}\nd={r['nearest_distance']:.3f}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7) Match each detected scene face against the query (`d1..dn`)

Objective steps 3, 5, 6, 7 happen here:
- Extract signatures for each scene crop
- Compute distances to query signature
- Select minimum distance
- Draw green bounding box + ID

In [ ]:
# -----------------------------
# IDENTIFY QUERY IN SCENE
# -----------------------------
# Optional decision threshold for cosine distance:
# lower = more similar. Tune this based on your dataset quality.
MATCH_THRESHOLD = 0.35

results = []

for idx, item in enumerate(scene_faces):
    # Convert float RGB crop [0,1] -> uint8 RGB [0,255]
    crop_rgb = (item["face"] * 255).astype(np.uint8)

    # DeepFace.represent accepts image arrays too.
    try:
        rep = DeepFace.represent(
            img_path=crop_rgb,
            model_name=MODEL_NAME,
            detector_backend="skip",   # already a face crop
            enforce_detection=False
        )
        if len(rep) == 0:
            continue
        scene_emb = np.array(rep[0]["embedding"], dtype=np.float32)
    except Exception:
        continue

    # Distance from query embedding to this detected face embedding.
    d_query = cosine_distance(query_embedding, scene_emb)

    # Also estimate identity by nearest gallery signature (optional but useful for ID label).
    gallery_distances = gallery_df["embedding"].apply(lambda e: cosine_distance(scene_emb, e))
    best_gallery_idx = int(gallery_distances.idxmin())
    best_gallery_distance = float(gallery_distances.loc[best_gallery_idx])
    predicted_id = gallery_df.loc[best_gallery_idx, "identity"]

    area = item["facial_area"]
    results.append({
        "face_index": idx,
        "query_distance": d_query,
        "predicted_id": predicted_id,
        "gallery_distance": best_gallery_distance,
        "x": int(area["x"]),
        "y": int(area["y"]),
        "w": int(area["w"]),
        "h": int(area["h"]),
    })

results_df = pd.DataFrame(results).sort_values("query_distance").reset_index(drop=True)
print("Detected faces with valid embeddings:", len(results_df))
display(results_df.head(10))

if len(results_df) == 0:
    raise RuntimeError("No valid face embeddings extracted from scene detections.")

best = results_df.iloc[0]
is_match = best["query_distance"] <= MATCH_THRESHOLD

print("\nBest candidate:")
print(best)
print(f"Match decision (query_distance <= {MATCH_THRESHOLD}):", is_match)

# Draw result on original scene image.
out = scene_bgr.copy()

x, y, w, h = int(best["x"]), int(best["y"]), int(best["w"]), int(best["h"])

# Green bounding box as requested in objective.
cv2.rectangle(out, (x, y), (x + w, y + h), (0, 255, 0), 3)

label = f"PredID={best['predicted_id']} | qDist={best['query_distance']:.3f}"
if not is_match:
    label = "NO CONFIDENT MATCH | " + label

# Label background for readability.
(text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
cv2.rectangle(out, (x, max(0, y - 28)), (x + text_w + 8, y), (0, 255, 0), -1)
cv2.putText(out, label, (x + 4, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2, cv2.LINE_AA)

# Save the final annotated image so the output is available as a file too.
output_image_path = str(PROJECT_ROOT / "output_query_localization.jpg")
cv2.imwrite(output_image_path, out)
print(f"Annotated output image saved to: {output_image_path}")

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
plt.title("Final Output: localized and identified query face")
plt.axis("off")
plt.show()

## 8) (Optional) Evaluate multiple scene images

This section loops over many scene images and reports whether the target identity was found. It is useful for your report (quantitative evidence).

In [ ]:
# -----------------------------
# SIMPLE BATCH EVALUATION OVER SCENE IMAGES
# -----------------------------
scene_paths = meta_df[meta_df["split"] == "scene_pool"]["path"].tolist()
batch_rows = []

for sp in tqdm(scene_paths, desc="Evaluating scenes"):
    try:
        scene_img = cv2.imread(sp)
        if scene_img is None:
            raise FileNotFoundError(f"Could not read scene image: {sp}")

        # Reuse robust detection helper to avoid full-frame fallback boxes.
        faces = detect_scene_faces_robust(scene_img)

        best_qdist = 999.0
        best_pid = None

        for item in faces:
            crop_rgb = (item["face"] * 255).astype(np.uint8)
            rep = DeepFace.represent(
                img_path=crop_rgb,
                model_name=MODEL_NAME,
                detector_backend="skip",
                enforce_detection=False
            )
            if len(rep) == 0:
                continue

            emb = np.array(rep[0]["embedding"], dtype=np.float32)
            qd = cosine_distance(query_embedding, emb)

            if qd < best_qdist:
                # Predict ID by nearest gallery embedding.
                gdist = gallery_df["embedding"].apply(lambda e: cosine_distance(emb, e))
                best_idx = int(gdist.idxmin())
                best_pid = gallery_df.loc[best_idx, "identity"]
                best_qdist = qd

        batch_rows.append({
            "scene_path": sp,
            "scene_token": Path(sp).stem.split("_")[0].upper(),
            "best_query_distance": best_qdist,
            "best_predicted_id": best_pid,
            "match": best_qdist <= MATCH_THRESHOLD,
        })
    except Exception as e:
        batch_rows.append({
            "scene_path": sp,
            "scene_token": Path(sp).stem.split("_")[0].upper(),
            "best_query_distance": np.nan,
            "best_predicted_id": None,
            "match": False,
            "error": str(e),
        })

batch_df = pd.DataFrame(batch_rows).sort_values("best_query_distance", na_position="last")
display(batch_df.head(20))

## 9) (Optional) Video pipeline for drone footage

If you have a drone video of a crowd, this function processes frames and draws the best match in each frame.

This extends the same objective pipeline from still images to video.

In [ ]:
# -----------------------------
# OPTIONAL: PROCESS VIDEO FRAMES
# -----------------------------
def process_video(
    input_video_path: str,
    output_video_path: str,
    query_embedding: np.ndarray,
    gallery_df: pd.DataFrame,
    model_name: str = MODEL_NAME,
    match_threshold: float = MATCH_THRESHOLD,
    frame_stride: int = 3
):
    """
    Process a video and annotate best query match in each processed frame.

    frame_stride:
        - 1 = process every frame (slower)
        - 2/3/4 = process every Nth frame (faster)
    """
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {input_video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_video_path, fourcc, fps if fps > 0 else 20.0, (width, height))

    frame_id = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_id % frame_stride != 0:
            writer.write(frame)
            frame_id += 1
            continue

        # Save temporary RGB for DeepFace detection path compatibility.
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        try:
            faces = DeepFace.extract_faces(
                img_path=frame_rgb,
                detector_backend="retinaface",
                enforce_detection=False,
                align=True
            )
        except Exception:
            faces = []

        best_item = None
        best_qdist = 999.0
        best_pid = None

        for item in faces:
            crop_rgb = (item["face"] * 255).astype(np.uint8)
            try:
                rep = DeepFace.represent(
                    img_path=crop_rgb,
                    model_name=model_name,
                    detector_backend="skip",
                    enforce_detection=False
                )
                if len(rep) == 0:
                    continue
                emb = np.array(rep[0]["embedding"], dtype=np.float32)
            except Exception:
                continue

            qd = cosine_distance(query_embedding, emb)
            if qd < best_qdist:
                gdist = gallery_df["embedding"].apply(lambda e: cosine_distance(emb, e))
                gid = int(gdist.idxmin())
                best_pid = gallery_df.loc[gid, "identity"]
                best_qdist = qd
                best_item = item

        # Draw only best candidate to match project objective (single target localization).
        if best_item is not None:
            area = best_item["facial_area"]
            x, y, w, h = int(area["x"]), int(area["y"]), int(area["w"]), int(area["h"])
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            text = f"ID={best_pid} | qDist={best_qdist:.3f}"
            if best_qdist > match_threshold:
                text = "LOW CONF: " + text
            cv2.putText(frame, text, (x, max(20, y - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        writer.write(frame)
        frame_id += 1

    cap.release()
    writer.release()
    print(f"Annotated video saved to: {output_video_path}")

# Example usage (uncomment and edit paths):
# process_video(
#     input_video_path="path/to/drone_video.mp4",
#     output_video_path="annotated_drone_video.mp4",
#     query_embedding=query_embedding,
#     gallery_df=gallery_df,
#     frame_stride=3
# )

## 10) Objective checklist (mapping)

- **Task 1 (Data collection):** Dataset discovered/extracted and explored (`open_data_set`)
- **Task 2.1:** Detect all faces in input image (`DeepFace.extract_faces`)
- **Task 2.2:** Crop detected faces (returned by extractor)
- **Task 2.3:** Extract signatures for all cropped faces (`DeepFace.represent`)
- **Task 2.4:** Extract query signature (`query_embedding`)
- **Task 2.5:** Compute distances (`cosine_distance`)
- **Task 2.6:** Select minimum distance face (`results_df.sort_values`)
- **Task 2.7:** Draw green bounding box with predicted ID (`cv2.rectangle`, `cv2.putText`)

You now have a complete, report-ready implementation aligned with your project objectives.